<a href="https://colab.research.google.com/github/azore67/pcl-semeval-coursework/blob/main/train_best_model_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
!pip -q install transformers scikit-learn accelerate

import os
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import f1_score, precision_recall_curve, confusion_matrix, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

set_seed(42)
print("CUDA:", torch.cuda.is_available())

CUDA: True


In [34]:
DPM_PATH = "data/dontpatronizeme_pcl.tsv"
TRAIN_IDS_PATH = "data/train_semeval_parids-labels.csv"
DEV_IDS_PATH = "data/dev_semeval_parids-labels.csv"
TEST_PATH = "data/task4_test.tsv"

for p in [DPM_PATH, TRAIN_IDS_PATH, DEV_IDS_PATH, TEST_PATH]:
    assert os.path.exists(p), f"Missing file: {p}"

dpm_raw = pd.read_csv(DPM_PATH, sep="\t", engine="python", skiprows=4, header=None)
print("DPM shape:", dpm_raw.shape)
assert dpm_raw.shape[1] == 6

dpm_raw.columns = ["par_id", "art_id", "keyword", "country", "text", "label"]
dpm_raw["par_id"] = pd.to_numeric(dpm_raw["par_id"], errors="raise").astype(int)
dpm_raw["label"] = pd.to_numeric(dpm_raw["label"], errors="raise").astype(int)
dpm_raw["binary_label"] = (dpm_raw["label"] != 0).astype(int)

dpm = dpm_raw[["par_id", "text", "binary_label"]].copy()

train_ids = pd.read_csv(TRAIN_IDS_PATH)
dev_ids = pd.read_csv(DEV_IDS_PATH)
train_ids["par_id"] = train_ids["par_id"].astype(int)
dev_ids["par_id"] = dev_ids["par_id"].astype(int)

train_df = train_ids[["par_id"]].merge(dpm, on="par_id", how="left")
dev_df   = dev_ids[["par_id"]].merge(dpm, on="par_id", how="left")

if dev_df["text"].isna().sum() > 0:
    missing_ids = dev_df.loc[dev_df["text"].isna(), "par_id"].tolist()
    print("Dropping missing dev rows:", len(missing_ids), "Missing:", missing_ids)
    dev_df = dev_df.dropna(subset=["text"]).reset_index(drop=True)

print("Train:", len(train_df), "Dev:", len(dev_df))
print("Train counts:\n", train_df["binary_label"].value_counts())
print("Dev counts:\n", dev_df["binary_label"].value_counts())

DPM shape: (10469, 6)
Dropping missing dev rows: 1 Missing: [8640]
Train: 8375 Dev: 2093
Train counts:
 binary_label
0    6825
1    1550
Name: count, dtype: int64
Dev counts:
 binary_label
0    1703
1     390
Name: count, dtype: int64


In [35]:

pos_df = train_df[train_df["binary_label"] == 1]
neg_df = train_df[train_df["binary_label"] == 0].sample(
    n=min(len(train_df[train_df["binary_label"] == 0]), 2 * len(pos_df)),
    random_state=42
)

train_bal = pd.concat([pos_df, neg_df]).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Balanced train size:", len(train_bal))
print(train_bal["binary_label"].value_counts())

Balanced train size: 4650
binary_label
0    3100
1    1550
Name: count, dtype: int64


In [36]:
MODEL_NAME = "roberta-base"
MAX_LEN = 160

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(
            list(map(str, texts)),
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        )
        self.labels = None if labels is None else list(map(int, labels))

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = TextDataset(train_bal["text"].tolist(), train_bal["binary_label"].tolist())
dev_dataset   = TextDataset(dev_df["text"].tolist(), dev_df["binary_label"].tolist())

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Train dataset:", len(train_dataset), "Dev dataset:", len(dev_dataset))

Train dataset: 4650 Dev dataset: 2093


In [37]:

counts = train_df["binary_label"].value_counts().sort_index()
neg = int(counts.get(0, 0))
pos = int(counts.get(1, 0))
class_weights = torch.tensor([1.0, neg / max(pos, 1)], dtype=torch.float)
print("Class weights:", class_weights.tolist())

class WeightedTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self.class_weights.to(device=logits.device, dtype=logits.dtype)
        loss_fn = nn.CrossEntropyLoss(weight=w)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)
    return {"f1_pos@0.5": f1_score(labels, preds, pos_label=1)}

Class weights: [1.0, 4.403225898742676]


In [38]:
TOTAL_EPOCHS = 5

args = TrainingArguments(
    output_dir="BestModel/checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 4,
    per_device_eval_batch_size=32 if torch.cuda.is_available() else 8,
    num_train_epochs=1,
    weight_decay=0.01,
    do_eval=True,
    logging_steps=50,
    fp16=False,
    report_to="none",
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

for e in range(TOTAL_EPOCHS):
    trainer.train()
    m = trainer.evaluate()
    print(f"\nAfter epoch {e+1}/{TOTAL_EPOCHS} -> Dev F1 (pos @0.5): {m.get('eval_f1_pos@0.5'):.4f}\n")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.612139
100,0.479175
150,0.491060
200,0.422718
250,0.441744


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


After epoch 1/5 -> Dev F1 (pos @0.5): 0.5562



...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


After epoch 4/5 -> Dev F1 (pos @0.5): 0.5955



Step,Training Loss
50,0.155663
100,0.245127
150,0.362990
200,0.286078
250,0.377738


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


After epoch 5/5 -> Dev F1 (pos @0.5): 0.6094



In [39]:
import numpy as np
import torch
from sklearn.metrics import precision_recall_curve, f1_score, confusion_matrix, classification_report

dev_out = trainer.predict(dev_dataset)
dev_logits = dev_out.predictions
dev_probs = torch.softmax(torch.tensor(dev_logits), dim=-1).numpy()[:, 1]
dev_labels = np.array(dev_df["binary_label"].astype(int).tolist())

prec, rec, thr = precision_recall_curve(dev_labels, dev_probs)
f1s = (2 * prec * rec) / (prec + rec + 1e-12)
best_i = int(np.nanargmax(f1s))

best_thr = 0.5 if best_i >= len(thr) else float(thr[best_i])
dev_preds_best = (dev_probs >= best_thr).astype(int)
best_f1 = f1_score(dev_labels, dev_preds_best, pos_label=1)

print(f"Best threshold: {best_thr:.4f}")
print(f"Dev F1 (pos) @ best threshold: {best_f1:.4f}")
print("\nConfusion matrix:\n", confusion_matrix(dev_labels, dev_preds_best))
print("\nReport:\n", classification_report(dev_labels, dev_preds_best, digits=4))

Best threshold: 0.7753
Dev F1 (pos) @ best threshold: 0.6284

Confusion matrix:
 [[1460  243]
 [ 100  290]]

Report:
               precision    recall  f1-score   support

           0     0.9359    0.8573    0.8949      1703
           1     0.5441    0.7436    0.6284       390

    accuracy                         0.8361      2093
   macro avg     0.7400    0.8005    0.7616      2093
weighted avg     0.8629    0.8361    0.8452      2093



In [40]:
with open("dev.txt", "w") as f:
    for p in dev_preds_best:
        f.write(str(int(p)) + "\n")

print("Wrote dev.txt with", len(dev_preds_best), "lines")

Wrote dev.txt with 2093 lines


In [41]:
import pandas as pd

TEST_PATH = "data/task4_test.tsv"
test = pd.read_csv(TEST_PATH, sep="\t", engine="python")

# robust text column selection
if "text" in test.columns:
    test_texts = test["text"].astype(str).tolist()
elif "paragraph" in test.columns:
    test_texts = test["paragraph"].astype(str).tolist()
else:
    test_texts = test.iloc[:, -1].astype(str).tolist()

test_dataset = TextDataset(test_texts, labels=None)

test_logits = trainer.predict(test_dataset).predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()[:, 1]
test_preds = (test_probs >= best_thr).astype(int)

with open("test.txt", "w") as f:
    for p in test_preds:
        f.write(str(int(p)) + "\n")

print("Wrote test.txt with", len(test_preds), "lines")
print("First 10 test preds:", test_preds[:10].tolist())

Wrote test.txt with 3831 lines
First 10 test preds: [1, 0, 0, 0, 1, 1, 0, 1, 0, 0]


In [42]:
import os

save_dir = "BestModel/model"
os.makedirs(save_dir, exist_ok=True)
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved model + tokenizer to:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model + tokenizer to: BestModel/model


In [43]:
dev_view = dev_df.copy()
dev_view["prob_pcl"] = dev_probs
dev_view["pred"] = dev_preds_best
dev_view["gold"] = dev_labels

false_pos = dev_view[(dev_view["pred"] == 1) & (dev_view["gold"] == 0)].sort_values("prob_pcl", ascending=False)
false_neg = dev_view[(dev_view["pred"] == 0) & (dev_view["gold"] == 1)].sort_values("prob_pcl", ascending=True)

print("Top false positives (model very confident but wrong):")
display(false_pos[["par_id", "prob_pcl", "text"]].head(8))

print("\nTop false negatives (missed PCL):")
display(false_neg[["par_id", "prob_pcl", "text"]].head(8))

Top false positives (model very confident but wrong):


,par_id,prob_pcl,text
568,8787,0.996739,Bishop Moss told Bishop Golding that the leade...
1784,10129,0.996128,"In other words , Waitiki 's now settled squatt..."
1851,10200,0.996012,She also praised the efforts to stabilise the ...
1661,9995,0.995875,"In his a New Year message , stated that Presid..."
1749,10090,0.995652,""" We have identified extremely poor families ,..."
769,9011,0.995515,ATD wants to uplift the self-esteem of MNC res...
957,9215,0.995508,My conclusion is that ours is a country and a ...
1260,9543,0.995390,"Many of the children are from poor families , ..."



Top false negatives (missed PCL):


,par_id,prob_pcl,text
1730,10071,0.002235,She is among the about 300 families in Bisan-h...
1509,9828,0.002235,Universities and colleges in several US states...
18,6234,0.002331,The World Health Organization did not give a r...
1415,9721,0.003020,About 10 residents of a compound house near co...
274,8465,0.003114,"Cecilia Acuin , chief science research special..."
1006,9268,0.003247,""" How many more kidnappings or illegal immigra..."
896,9152,0.003251,Stories about sexual violence against refugee ...
602,8825,0.003327,In an exhibition match after the tournament fi...


In [44]:
import pandas as pd

TEST_PATH = "data/task4_test.tsv"
test = pd.read_csv(TEST_PATH, sep="\t", engine="python", header=None)  # don't guess headers
print("Raw test shape:", test.shape)
test.head()

Raw test shape: (3832, 5)


,0,1,2,3,4
0,t_0,@@7258997,vulnerable,us,"In the meantime , conservatives are working to..."
1,t_1,@@16397324,women,pk,In most poor households with no education chil...
2,t_2,@@16257812,migrant,ca,The real question is not whether immigration i...
3,t_3,@@3509652,migrant,gb,"In total , the country 's immigrant population..."
4,t_4,@@477506,vulnerable,ca,"Members of the church , which is part of Ken C..."


In [52]:
!git add dev.txt test.txt BestModel/model

In [53]:
!git commit -m "Add final dev/test predictions and saved model"

[main dc7575f] Add final dev/test predictions and saved model
 6 files changed, 256331 insertions(+)
 create mode 100644 BestModel/model/config.json
 create mode 100644 BestModel/model/model.safetensors
 create mode 100644 BestModel/model/tokenizer.json
 create mode 100644 BestModel/model/tokenizer_config.json
 create mode 100644 dev.txt
 create mode 100644 test.txt


In [54]:
!git push

fatal: could not read Username for 'https://github.com': No such device or address


In [51]:
!git config --global user.email "atharva.zore2005@gmail.com"
!git config --global user.name "a.zore67"
